In [8]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [9]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH= "telecom_guide.pdf"

loader= PyPDFLoader(PDF_PATH)
pages= loader.load()
print(f"loaded {len(pages)} pages from the the pdf")
print("\n---First 500 Pages from the pdf---\n")
print(pages[0].page_content[:500])

C:\Users\gargk\AppData\Local\Temp\ipykernel_15332\4247998400.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


loaded 9 pages from the the pdf

---First 500 Pages from the pdf---

Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [9]:
print(pages[2].page_content[:200])

Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approac


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter= RecursiveCharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap= 100,
    separators= ["\n\n", "\n", ".", " "]
)

chunks= splitter.split_documents(pages)

total_char = sum(len(x.page_content) for x in chunks)
avg_char = total_char / len(chunks)
print("Total characters:", total_char)
print("Average per chunk:", avg_char)

print(len(chunks))
len(pages)

Total characters: 18596
Average per chunk: 413.24444444444447
18596
45


9

In [19]:
chunks[1].page_content

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.'

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
vector_store= Chroma.from_documents(chunks, embeddings)
print(f"vector store ready. {vector_store._collection.count()} vectors stored")


C:\Users\gargk\AgenticAIProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1992.07it/s]


vector store ready. 45 vectors stored


In [12]:
stored = vector_store._collection.get(include=["embeddings"])
print(len(stored["embeddings"]))
print(len(stored["embeddings"][11]))
#print(stored["embeddings"][10])   # first embedding vector

45
384


In [13]:
# Token count for chunks
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

# Load tokenizer for the same model
tokenizer = AutoTokenizer.from_pretrained(model_name)

total_tokens = sum(len(tokenizer.encode(x.page_content)) for x in chunks)
print("Total tokens:", total_tokens)

Total tokens: 3961


In [14]:
retriever= vector_store.as_retriever(search_kwargs={"k":2})

test_query= "What is VoltE and how does it improve the call quality?"
retrieved= retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"---Chunks {i}---")
    print(type(retriever))
    print(doc.page_content)

---Chunks 1---
<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Options > Voice & Data and select LTE. If the option is absent
---Chunks 2---
<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS 

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)
    
SYSTEM_PROMPT= """\
you are helpful telecom assistant.
Answer the question using only the context provided below.
If the context doesn't contain enough information, say so clearly.

context:
{context}
"""
    
prompt= ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

llm= ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    reasoning_format="parsed",
)

chain= (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain assembled")

RAG chain assembled


In [30]:
question= "How does international roaming work and what charges can I expect?"
print(f"Q: {question}\n")
print(f"A: {chain.invoke(question)}")

Q: How does international roaming work and what charges can I expect?

A: Based on the provided context, here is how international roaming works and the associated charges:

**How It Works:**
*   **Connection:** When you travel outside your home network's coverage area, your device connects to a partner network in the visited country.
*   **Authentication:** The visited network authenticates your device via an inter-operator signalling protocol.
*   **Activation:** Roaming must be enabled on your account **before** departure. You can activate it by:
    *   Toggling it in the MyTelecom app under **Plan & Services > International Roaming**.
    *   Calling **611**.
    *   *Note:* Network-level activation takes time (the specific duration is cut off in the provided text).

**Charges:**
Charges vary by destination zone:
*   **Zone A:** (Countries like Japan, Singapore) – Moderate rates.
*   **Zone B:** (Not explicitly listed in the snippet, but implied to have higher rates than Zone A).
